In [1]:
import os  

In [2]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow\\notebooks'

In [3]:
os.chdir("../")

In [4]:
%pwd

'd:\\ML-Projects\\End-to-end-Machine-Learning-World-Development-Measurement-Clustering-Analysis-with-MLFlow'

In [60]:
# Components of data validation
import pandas as pd

In [61]:
data = pd.read_excel("artifacts/data_ingestion/World_development_mesurement.xlsx")   
data.head()

,Birth Rate,Business Tax Rate,CO2 Emissions,Country,Days to Start Business,Ease of Business,Energy Usage,GDP,Health Exp % GDP,Health Exp/Capita,...,Life Expectancy Male,Mobile Phone Usage,Number of Records,Population 0-14,Population 15-64,Population 65+,Population Total,Population Urban,Tourism Inbound,Tourism Outbound
0,0.020,NaN,87931.0,Algeria,NaN,NaN,26998.0,"$54,790,058,957",0.035,$60,...,67.0,0.0,1,0.342,0.619,0.039,31719449,0.599,"$102,000,000","$193,000,000"
1,0.050,NaN,9542.0,Angola,NaN,NaN,7499.0,"$9,129,594,819",0.034,$22,...,44.0,0.0,1,0.476,0.499,0.025,13924930,0.324,"$34,000,000","$146,000,000"
2,0.043,NaN,1617.0,Benin,NaN,NaN,1983.0,"$2,359,122,303",0.043,$15,...,53.0,0.0,1,0.454,0.517,0.029,6949366,0.383,"$77,000,000","$50,000,000"
3,0.027,NaN,4276.0,Botswana,NaN,NaN,1836.0,"$5,788,311,645",0.047,$152,...,49.0,0.1,1,0.383,0.587,0.029,1755375,0.532,"$227,000,000","$209,000,000"
4,0.046,NaN,1041.0,Burkina Faso,NaN,NaN,NaN,"$2,610,959,139",0.051,$12,...,49.0,0.0,1,0.468,0.505,0.028,11607944,0.178,"$23,000,000","$30,000,000"


In [62]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2704 entries, 0 to 2703
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Birth Rate              2585 non-null   float64
 1   Business Tax Rate       1423 non-null   object 
 2   CO2 Emissions           2125 non-null   float64
 3   Country                 2704 non-null   object 
 4   Days to Start Business  1718 non-null   float64
 5   Ease of Business        185 non-null    float64
 6   Energy Usage            1785 non-null   float64
 7   GDP                     2494 non-null   object 
 8   Health Exp % GDP        2395 non-null   float64
 9   Health Exp/Capita       2395 non-null   object 
 10  Hours to do Tax         1416 non-null   float64
 11  Infant Mortality Rate   2444 non-null   float64
 12  Internet Usage          2531 non-null   float64
 13  Lending Interest        1880 non-null   float64
 14  Life Expectancy Female  2568 non-null   

In [63]:
data.isnull().sum()

Birth Rate                 119
Business Tax Rate         1281
CO2 Emissions              579
Country                      0
Days to Start Business     986
Ease of Business          2519
Energy Usage               919
GDP                        210
Health Exp % GDP           309
Health Exp/Capita          309
Hours to do Tax           1288
Infant Mortality Rate      260
Internet Usage             173
Lending Interest           824
Life Expectancy Female     136
Life Expectancy Male       136
Mobile Phone Usage         167
Number of Records            0
Population 0-14            220
Population 15-64           220
Population 65+             220
Population Total             0
Population Urban            26
Tourism Inbound            368
Tourism Outbound           471
dtype: int64

In [64]:
data.shape

(2704, 25)

In [65]:
data.columns

Index(['Birth Rate', 'Business Tax Rate', 'CO2 Emissions', 'Country',
       'Days to Start Business', 'Ease of Business', 'Energy Usage', 'GDP',
       'Health Exp % GDP', 'Health Exp/Capita', 'Hours to do Tax',
       'Infant Mortality Rate', 'Internet Usage', 'Lending Interest',
       'Life Expectancy Female', 'Life Expectancy Male', 'Mobile Phone Usage',
       'Number of Records', 'Population 0-14', 'Population 15-64',
       'Population 65+', 'Population Total', 'Population Urban',
       'Tourism Inbound', 'Tourism Outbound'],
      dtype='object')

In [81]:
## Preparing Entity

from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: str
    all_schema: dict
    number_of_columns: int


In [82]:
## Configuration

from wdmproject.constants import *
from wdmproject.utils.common import read_yaml, create_directories

In [83]:
## Configuration manager class
class ConfigurationManager:
    def __init__(
        self,
        config_filepath: Path = CONFIG_FILE_PATH,
        params_filepath: Path = PARAMS_FILE_PATH,
        schema_filepath: Path = SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])

    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        schema_columns = self.schema.COLUMNS
        number_of_columns = self.schema.NUMBER_OF_COLUMNS

        create_directories([config.root_dir])
        
        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            unzip_data_dir=Path(config.unzip_data_dir),
            STATUS_FILE=Path(config.STATUS_FILE),
            all_schema=schema_columns,
            number_of_columns=number_of_columns,
        )
        
        return data_validation_config

In [84]:
import os
from wdmproject import logger

In [85]:
class DataValidation:
    def __init__(self, config: DataValidationConfig):
        self.config = config

    ## Checking wheather all the columns in dataset matches with schema
    def validate_all_columns(self) -> bool:
        try:

            data = pd.read_excel(self.config.unzip_data_dir)

            validation_status = None
            
            ## First checking the columns in dataset with expected columns in schema
            if len(data.columns) != self.config.number_of_columns:
                validation_status = False
                with open(self.config.STATUS_FILE, 'w') as f:
                    f.write(f"Data Validation Failed: Expected {self.config.number_of_columns} columns, but found {len(data.columns)} columns.\n")
                logger.error(f"Expected {self.config.number_of_columns} columns, but found {len(data.columns)} columns.")
                return validation_status
            
            ## IF columns lenght matches move forward to check the cloumns with the dtypes

            expected_columns = set(self.config.all_schema.keys())
            actual_columns = set(data.columns)
        
            missing_columns = expected_columns - actual_columns
            extra_columns = actual_columns - expected_columns
        
            ## Checking missing columns        
            if missing_columns:

                validation_status = False
                with open(self.config.STATUS_FILE, 'w') as f:
                    f.write(f"Data Validation Failed: Missing columns: {missing_columns}\n")
                logger.error(f"Missing columns: {missing_columns}")
            else:
                validation_status = True
                with open(self.config.STATUS_FILE, 'w') as f:
                    f.write("Data Validation Passed: All expected columns are present.\n")

            if extra_columns:
                logger.warning(f"Extra columns: {extra_columns}")

            ## Validating Datatypes
            dtype_mismatch = []

            for column, expected_dtype in self.config.all_schema.items():

                if column in data.columns:

                    actual_dtype = str(data[column].dtype)

                    if actual_dtype != expected_dtype:

                        dtype_mismatch.append(
                            f"{column}: expected {expected_dtype}, got {actual_dtype}"
                        )

            
            if dtype_mismatch:
                validation_status = False
                logger.error(f"Datatype Mismatch: {dtype_mismatch}")

            ## Validation
            with open(self.config.STATUS_FILE, 'w') as f:

                if validation_status:
                    f.write('Validation Passed Successfully. \n')
                else:
                    f.write('Validation Failed Unfortunately. \n')
            
                if missing_columns:
                    f.write(f"Missing Columns : {missing_columns}\n")
                else:
                    f.write(f"No missing columns found!\n")

                if dtype_mismatch:
                    f.write(f"Datatype Mismatch: {dtype_mismatch}\n")
                else:
                    f.write(f"All the datatypes matched with the schema.\n")

        
            return validation_status 
        
        except Exception as e:
            logger.error(f"Data Validation Error: {e}")
            raise e

In [87]:
## Data Validation Pipeline
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(config=data_validation_config)
    data_validation.validate_all_columns()
    logger.info(f"Data Validation Pipeline Completed Successfully.")
except Exception as e:
    logger.error(f"Data Validation Pipeline Failed: {e}")
    raise e

[2026-03-09 17:42:01,636: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-09 17:42:01,640: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-09 17:42:01,651: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-03-09 17:42:01,654: INFO: common: created directory at: artifacts]
[2026-03-09 17:42:01,657: INFO: common: created directory at: artifacts/data_validation]
[2026-03-09 17:42:04,384: INFO: 2267731337: Data Validation Pipeline Completed Successfully.]
